In [2]:
!pip install catboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 27.1 MB/s eta 0:00:00


https://cups.online/ru/training/4/tasks/2380

In [3]:
import pandas as pd
import numpy as np
from catboost import Pool, CatBoostClassifier
import lightgbm as lgb
from pathlib import Path
import hashlib
import gc
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

In [5]:
base_dir = Path('.')
train = pd.read_parquet(base_dir / 'train.parquet')
test  = pd.read_parquet(base_dir / 'test.parquet')
sample_submission = pd.read_csv(base_dir / 'sample.csv')
video_stat = pd.read_parquet(base_dir / 'video_stat.parquet')

In [6]:
gc.collect()

105

In [7]:
%whos

Variable                Type         Data/Info
----------------------------------------------
CatBoostClassifier      type         <class 'catboost.core.CatBoostClassifier'>
Path                    type         <class 'pathlib.Path'>
Pool                    type         <class 'catboost.core.Pool'>
base_dir                PosixPath    .
classification_report   function     <function classification_<...>report at 0x7edbc70f44a0>
gc                      module       <module 'gc' (built-in)>
hashlib                 module       <module 'hashlib' from '/<...>b/python3.12/hashlib.py'>
lgb                     module       <module 'lightgbm' from '<...>es/lightgbm/__init__.py'>
np                      module       <module 'numpy' from '/us<...>kages/numpy/__init__.py'>
pd                      module       <module 'pandas' from '/u<...>ages/pandas/__init__.py'>
sample_submission       DataFrame             Unnamed: 0  targ<...>1334132 rows x 2 columns]
test                    DataFrame        

In [8]:
test['dataset_type']  = 'test'
test['target']  = 0
train['target'] = 0
test['watchtime'] = 0
train['dataset_type'] = 'train'
train = pd.concat([train, test], axis=0)

In [9]:
combined_data = train.merge(video_stat, on='video_id', how='left')
del train
gc.collect()

16

In [10]:
del test

In [11]:
def extract_target(data):
  long_long_view = (data.v_duration > 5*60) & (data.watchtime >= .25 * data.v_duration)
  long_short_view = (data.v_duration <= 5*60) & (data.watchtime >= 30)
  data['target'] = np.where(long_long_view | long_short_view, 1, 0)
  data = data.drop(columns=['watchtime'])
  return data

In [12]:
combined_data = extract_target(combined_data)
# combined_data.head()

In [13]:
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder

def hash_to_string(string, num_buckets):
  return int(hashlib.md5(string.encode('utf-8')).hexdigest(), 16) % num_buckets

num_buckets_user_id  = 45000000
num_buckets_video_id = 600000
num_buckets_author_id = 120000

combined_data['user_id']  = combined_data['user_id'].apply(lambda x: hash_to_string(x, num_buckets_user_id))
combined_data['video_id'] = combined_data['video_id'].apply(lambda x: hash_to_string(x, num_buckets_video_id))
combined_data['author_id']= combined_data['author_id'].apply(lambda x: hash_to_string(x, num_buckets_author_id))

user_emb_layer = nn.Embedding(num_buckets_user_id, 16)
video_emb_layer = nn.Embedding(num_buckets_video_id, 16)
author_emb_layer = nn.Embedding(num_buckets_author_id, 16)

user_emb   = user_emb_layer(torch.tensor(combined_data['user_id'].values, dtype=torch.long)).detach().numpy()
video_emb  = video_emb_layer(torch.tensor(combined_data['video_id'].values, dtype=torch.long)).detach().numpy()
author_emb = author_emb_layer(torch.tensor(combined_data['author_id'].values, dtype=torch.long)).detach().numpy()

le_region = LabelEncoder()
le_city = LabelEncoder()

combined_data['region'] = le_region.fit_transform(combined_data['region'])
combined_data['city'] = le_city.fit_transform(combined_data['city'])

region_emb_layer = nn.Embedding(combined_data['region'].nunique(), 8)
city_emb_layer   = nn.Embedding(combined_data['city'].nunique(), 8)

region_emb = region_emb_layer(torch.tensor(combined_data['region'].values, dtype=torch.long)).detach().numpy()
city_emb   = city_emb_layer(torch.tensor(combined_data['region'].values, dtype=torch.long)).detach().numpy()

In [14]:
del user_emb_layer, video_emb_layer, author_emb_layer, region_emb_layer, city_emb_layer

In [15]:
gc.collect()

46

In [16]:
le_category = LabelEncoder()
combined_data['category_id'] = le_category.fit_transform(combined_data['category_id'])

category_emb_layer = nn.Embedding(combined_data['category_id'].nunique(), 8)
categroy_emb = category_emb_layer(torch.tensor(combined_data['category_id'].values, dtype=torch.long)).detach().numpy()

del category_emb_layer

In [17]:
combined_data['event_timestamp_day']   = combined_data['event_timestamp'].dt.day
combined_data['event_timestamp_week']  = combined_data['event_timestamp'].dt.day_of_week
combined_data['event_timestamp_month'] = combined_data['event_timestamp'].dt.month
combined_data['v_pub_day']             = combined_data['v_pub_datetime'].dt.day
combined_data['v_pub_week']            = combined_data['v_pub_datetime'].dt.day_of_week
combined_data['v_pub_month']           = combined_data['v_pub_datetime'].dt.month

combined_data = combined_data.drop(columns=['event_timestamp', 'v_pub_datetime'])

In [19]:
combined_data['title_length'] = combined_data['title'].apply(lambda x: len(x))
combined_data['description_length'] = combined_data['description'].apply(lambda x: len(x))

In [20]:
gc.collect()

0

In [22]:
combined_data = combined_data.drop(columns=['title', 'description'])

In [28]:
del num_buckets_author_id, num_buckets_user_id, num_buckets_video_id

In [31]:
combined_data.columns

Index(['user_id', 'region', 'city', 'video_id', 'target', 'dataset_type',
       'v_total_comments', 'v_year_views', 'v_month_views', 'v_week_views',
       'v_day_views', 'v_likes', 'v_dislikes', 'v_duration',
       'v_cr_click_like_7_days', 'v_cr_click_dislike_7_days',
       'v_cr_click_vtop_7_days', 'v_cr_click_long_view_7_days',
       'v_cr_click_comment_7_days', 'v_cr_click_like_30_days',
       'v_cr_click_dislike_30_days', 'v_cr_click_vtop_30_days',
       'v_cr_click_long_view_30_days', 'v_cr_click_comment_30_days',
       'v_cr_click_like_1_days', 'v_cr_click_dislike_1_days',
       'v_cr_click_vtop_1_days', 'v_cr_click_long_view_1_days',
       'v_cr_click_comment_1_days', 'v_is_hidden', 'v_is_deleted',
       'v_avg_watchtime_1_day', 'v_avg_watchtime_7_day',
       'v_avg_watchtime_30_day', 'v_frac_avg_watchtime_1_day_duration',
       'v_frac_avg_watchtime_7_day_duration',
       'v_frac_avg_watchtime_30_day_duration',
       'v_category_popularity_percent_7_days',
     

In [36]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

numerical_features = [
"v_cr_click_long_view_30_days", "v_cr_click_long_view_7_days", "v_duration", "v_likes",
"v_dislikes", "v_total_comments",
"v_year_views", 'title_length', 'description_length',
'v_month_views', "v_week_views", "v_day_views",
'v_cr_click_long_view_1_days', 'v_avg_watchtime_1_day'
]

numeric_mat = scaler.fit_transform(combined_data[numerical_features].values)

user_id_df = pd.DataFrame(user_emb, columns=[f'user_id_emb_{i}' for i in range(user_emb.shape[1])])
video_id_df = pd.DataFrame(video_emb, columns=[f'video_id_emb_{i}' for i in range(video_emb.shape[1])])
city_id_df = pd.DataFrame(city_emb, columns=[f'city_id_emb_{i}' for i in range(city_emb.shape[1])])
author_id_df = pd.DataFrame(author_emb, columns=[f'author_id_emb_{i}' for i in range(author_emb.shape[1])])
city_id_df = pd.DataFrame(city_emb, columns=[f'city_id_emb_{i}' for i in range(city_emb.shape[1])])

numeric_df = pd.DataFrame(numeric_mat, columns=numerical_features)
final_df = pd.concat([
    combined_data['dataset_type'],
    combined_data['target'],
    combined_data[['event_timestamp_day','event_timestamp_week', 'event_timestamp_month', 'v_pub_day','v_pub_week', 'v_pub_month']],
    numeric_df,
    user_id_df,
    video_id_df,
    city_id_df,
    author_id_df,
    city_id_df
], axis=1)

In [147]:
from sklearn.model_selection import train_test_split
X_train = final_df[final_df.dataset_type == 'train'].drop(columns=['dataset_type'])
X_test  = final_df[final_df.dataset_type == 'test'].drop(columns=['dataset_type'])

X_train, y_train = X_train.drop(columns=['target']), X_train['target']

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, train_size=.8, random_state=42)

In [148]:
del numeric_mat, video_stat

NameError: name 'numeric_mat' is not defined

In [149]:
gc.collect()

813

In [70]:
model = CatBoostClassifier(max_depth=8, iterations=500, l2_leaf_reg=6, verbose=100,
                           learning_rate=.05, grow_policy='SymmetricTree', task_type='GPU')
model.fit(Pool(X_train, y_train))

'              precision    recall  f1-score   support\n\n           0       0.74      0.57      0.65   5295943\n           1       0.69      0.83      0.75   6027708\n\n    accuracy                           0.71  11323651\n   macro avg       0.71      0.70      0.70  11323651\nweighted avg       0.71      0.71      0.70  11323651\n'

In [ ]:
X_val = X_val.loc[:, ~X_val.columns.duplicated()]

In [102]:
preds = model.predict(X_val)

print(classification_report(y_val, preds))

              precision    recall  f1-score   support

           0       0.74      0.57      0.65   1324131
           1       0.69      0.83      0.75   1506782

    accuracy                           0.71   2830913
   macro avg       0.71      0.70      0.70   2830913
weighted avg       0.71      0.71      0.70   2830913



In [76]:
# посмотрим на важность признаков катбуста
fi = model.get_feature_importance(prettified=True)

fi.head(76)

,Feature Id,Importances
0,v_duration,17.064602
1,v_cr_click_long_view_7_days,17.013117
2,v_cr_click_long_view_1_days,11.975870
3,v_cr_click_long_view_30_days,10.461567
4,v_year_views,5.633810
...,...,...
71,user_id_emb_4,0.001111
72,event_timestamp_day,0.000000
73,event_timestamp_week,0.000000
74,event_timestamp_month,0.000000


In [77]:
gc.collect()

672

In [85]:
X_test

,event_timestamp_day,event_timestamp_week,event_timestamp_month,v_pub_day,v_pub_week,v_pub_month,v_cr_click_long_view_30_days,v_cr_click_long_view_7_days,v_duration,v_dislikes,v_total_comments,v_year_views,title_length,description_length,v_month_views,v_week_views,v_day_views,v_cr_click_long_view_1_days,v_avg_watchtime_1_day,user_id_emb_0,user_id_emb_1,user_id_emb_2,user_id_emb_3,user_id_emb_4,user_id_emb_5,user_id_emb_6,user_id_emb_7,user_id_emb_8,user_id_emb_9,user_id_emb_10,user_id_emb_11,user_id_emb_12,user_id_emb_13,user_id_emb_14,user_id_emb_15,video_id_emb_0,video_id_emb_1,video_id_emb_2,video_id_emb_3,video_id_emb_4,...,video_id_emb_7,video_id_emb_8,video_id_emb_9,video_id_emb_10,video_id_emb_11,video_id_emb_12,video_id_emb_13,video_id_emb_14,video_id_emb_15,city_id_emb_1,city_id_emb_2,city_id_emb_3,city_id_emb_4,city_id_emb_5,city_id_emb_6,city_id_emb_7,author_id_emb_0,author_id_emb_1,author_id_emb_2,author_id_emb_3,author_id_emb_4,author_id_emb_5,author_id_emb_6,author_id_emb_7,author_id_emb_8,author_id_emb_9,author_id_emb_10,author_id_emb_11,author_id_emb_12,author_id_emb_13,author_id_emb_14,author_id_emb_15,city_id_emb_1,city_id_emb_2,city_id_emb_3,city_id_emb_4,city_id_emb_5,city_id_emb_6,city_id_emb_7,target
14154564,10,5,8,30,4,12,1.096767,1.096767,-0.461266,-0.198496,-0.058728,-0.136425,0.070508,-0.307014,0.011923,0.011923,0.011923,1.096767,-0.034496,1.547186,0.136097,1.208937,0.062568,-1.010262,0.995992,1.975547,0.950439,-0.432141,-1.252128,-0.851568,-0.757473,0.172839,-0.494822,-0.068772,0.361965,-1.326063,-0.326428,-0.997921,-0.952125,1.668391,...,1.635516,0.382862,0.085039,-0.240998,1.442390,0.287744,0.257481,0.407433,-0.237365,-1.768839,1.01721,0.557811,-1.114945,1.290511,1.20057,-0.0458,-0.790491,0.894816,-0.061883,-1.193625,0.381702,-0.428841,-0.428503,0.581902,0.391096,0.335656,-0.523360,0.515552,0.341824,0.677611,-0.142574,0.163317,-1.768839,1.01721,0.557811,-1.114945,1.290511,1.20057,-0.0458,0
14154565,10,5,8,30,4,12,1.120156,1.120156,-0.458389,-0.153678,-0.059016,-0.142078,-0.321753,-0.301970,0.000891,0.000891,0.000891,1.120156,-0.033786,1.547186,0.136097,1.208937,0.062568,-1.010262,0.995992,1.975547,0.950439,-0.432141,-1.252128,-0.851568,-0.757473,0.172839,-0.494822,-0.068772,0.361965,0.185909,-1.506796,1.378002,-0.334277,0.217691,...,0.320036,-0.732969,0.596445,2.339393,0.104787,-0.638309,0.678456,0.033691,0.607678,-1.768839,1.01721,0.557811,-1.114945,1.290511,1.20057,-0.0458,-0.790491,0.894816,-0.061883,-1.193625,0.381702,-0.428841,-0.428503,0.581902,0.391096,0.335656,-0.523360,0.515552,0.341824,0.677611,-0.142574,0.163317,-1.768839,1.01721,0.557811,-1.114945,1.290511,1.20057,-0.0458,0
14154566,10,5,8,30,4,12,1.264742,1.264742,-0.464962,-0.171605,-0.059100,-0.139762,0.288430,-0.049799,0.266488,0.266488,0.266488,1.264742,-0.033653,1.547186,0.136097,1.208937,0.062568,-1.010262,0.995992,1.975547,0.950439,-0.432141,-1.252128,-0.851568,-0.757473,0.172839,-0.494822,-0.068772,0.361965,-1.216936,-0.090473,-0.373238,0.569667,0.931805,...,-1.041074,-0.843950,-1.323101,0.183132,0.540932,-0.458341,-1.075647,0.182661,-0.357854,-1.768839,1.01721,0.557811,-1.114945,1.290511,1.20057,-0.0458,-0.790491,0.894816,-0.061883,-1.193625,0.381702,-0.428841,-0.428503,0.581902,0.391096,0.335656,-0.523360,0.515552,0.341824,0.677611,-0.142574,0.163317,-1.768839,1.01721,0.557811,-1.114945,1.290511,1.20057,-0.0458,0
14154567,10,5,8,30,4,12,1.140961,1.140961,-0.458463,-0.162642,-0.059083,-0.140764,-0.801182,-0.280115,-0.004329,-0.004329,-0.004329,1.140961,-0.033802,1.547186,0.136097,1.208937,0.062568,-1.010262,0.995992,1.975547,0.950439,-0.432141,-1.252128,-0.851568,-0.757473,0.172839,-0.494822,-0.068772,0.361965,0.627387,-0.160613,0.144297,0.425127,-0.076175,...,2.419905,-0.832487,-0.311694,0.945249,0.566366,0.458077,0.800631,1.918848,-0.821948,-1.768839,1.01721,0.557811,-1.114945,1.290511,1.20057,-0.0458,-0.790491,0.894816,-0.061883,-1.193625,0.381702,-0.428841,-0.428503,0.581902,0.391096,0.335656,-0.523360,0.515552,0.341824,0

In [107]:
# Оставляем только уникальные по названию колонки
X_test = X_test.loc[:, ~X_test.columns.duplicated()]
X_train = X_train.loc[:, ~X_train.columns.duplicated()]

In [92]:
final_preds = model.predict(X_train)

In [99]:
pd.DataFrame(final_preds, columns=['target']).to_csv('sub.csv', index=False)

In [95]:
sample_submission

,Unnamed: 0,target
0,0,0
1,1,1
2,2,1
3,3,1
4,4,0
...,...,...
1334127,1334127,1
1334128,1334128,1
1334129,1334129,1
1334130,1334130,1


In [105]:
gc.collect()

1219

In [109]:
lgbm = lgb.LGBMClassifier(
    num_leaves = 32,
    max_depth = 8,
    learning_rate = 0.05,
    n_estimators = 500,
    random_state = 42,
    device_type='gpu'
)

lgbm.fit(X_train, y_train)

preds = lgbm.predict(X_val)

print(classification_report(y_val, preds))

[LightGBM] [Info] Number of positive: 6027708, number of negative: 5295943
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 17119
[LightGBM] [Info] Number of data points in the train set: 11323651, number of used features: 76
[LightGBM] [Info] Using GPU Device: NVIDIA A100-SXM4-80GB, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 76 dense feature groups (820.73 MB) transferred to GPU in 0.387261 secs. 0 sparse feature groups
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.532311 -> initscore=0.129426
[LightGBM] [Info] Start training from score 0.129426
              precision    recall  f1-score   support

           0       0.74      0.57      0.65   1324131
           1       0.69      0.83      0.75   1506782

    accuracy                           0.71   2830913
   macro avg       0.71      0.70      0.70 

In [112]:
X_test = X_test.drop(columns=['target'])

In [133]:
lgbm_preds = lgbm.predict_proba(X_val)[:, 1]
cat_preds  = model.predict_proba(X_val)[:, 1]

In [134]:
preds = 0.7*cat_preds + 0.3*lgbm_preds

In [136]:
from sklearn.metrics import f1_score
thresholds = np.linspace(0, 1, 100)
scores = [f1_score(y_val, (preds > t).astype(int)) for t in thresholds]

best_threshold = thresholds[np.argmax(scores)]
print(f'Оптимальный порог: {best_threshold}')

final_labels = (preds > best_threshold).astype(int)

Оптимальный порог: 0.393939393939394


In [143]:
lgbm_preds = lgbm.predict_proba(X_test)[:, 1]
cat_preds  = model.predict_proba(X_test)[:, 1]
preds = 0.7*cat_preds + 0.3*lgbm_preds
final_preds = (preds > best_threshold).astype(int)

In [144]:
final_preds

array([1, 1, 1, ..., 0, 1, 1])

In [145]:
pd.DataFrame(final_preds, columns=['target']).to_csv('sub.csv', index=False)

Refitting on full data

In [ ]:
X_train = final_df[final_df.dataset_type == 'train'].drop(columns=['dataset_type'])
X_test  = final_df[final_df.dataset_type == 'test'].drop(columns=['dataset_type'])

X_train, y_train = X_train.drop(columns=['target']), X_train['target']

In [ ]:
model = CatBoostClassifier(max_depth=8, iterations=500, l2_leaf_reg=6, verbose=100,
                           learning_rate=.05, grow_policy='SymmetricTree', task_type='GPU')
model.fit(Pool(X_train, y_train))

gc.collect()

lgbm = lgb.LGBMClassifier(
    num_leaves = 32,
    max_depth = 8,
    learning_rate = 0.05,
    n_estimators = 500,
    random_state = 42,
    device_type='gpu'
)

lgbm.fit(X_train, y_train)

In [ ]:
best_threshold = 0.393939393939394
lgbm_preds = lgbm.predict_proba(X_test)[:, 1]
cat_preds  = model.predict_proba(X_test)[:, 1]
preds = 0.7*cat_preds + 0.3*lgbm_preds
final_preds = (preds > best_threshold).astype(int)

pd.DataFrame(final_preds, columns=['target']).to_csv('final_sub.csv', index=False)